# 1.08 多层感知机（MLP）与训练技巧

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/01_prerequisites/1.08_mlp_and_training.ipynb)


## 为什么要学

MLP 是最朴素的神经网络，但它无处不在：

- Transformer 里每一层都有一个 MLP（FFN）；
- **RLT 的 actor 和 critic 就是 MLP**（`rlt_mlp_policy.py`）；
- 扩散/流匹配策略里，把时间 $t$ 编码进网络的 `time_mlp` 也是 MLP。

本课还会讲"让训练稳定"的一整套技巧：**激活函数、初始化、归一化、正则化、优化器、学习率调度、梯度裁剪**。openpi 的训练配置里，这些全部都有。参考：[d2l 第 4 章 多层感知机](https://zh.d2l.ai/chapter_multilayer-perceptrons/index.html)、[d2l 第 11 章 优化算法](https://zh.d2l.ai/chapter_optimization/index.html)。

---

## 1. 为什么需要非线性？

两个线性层叠加 $W_2(W_1\mathbf{x}) = (W_2W_1)\mathbf{x}$ 还是线性的。中间插入一个**非线性激活函数** $\sigma$：

$$
\mathbf{h} = \sigma(W_1\mathbf{x} + \mathbf{b}_1),\qquad \hat{\mathbf{y}} = W_2\mathbf{h} + \mathbf{b}_2
$$

理论上（万能近似定理），一个足够宽的隐藏层就能逼近任意连续函数。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

x = torch.linspace(-4, 4, 200)
acts = {"ReLU": F.relu, "GELU (Gemma/ViT)": F.gelu, "SiLU/Swish (pi0 time MLP)": F.silu, "Tanh": torch.tanh}
plt.figure(figsize=(5, 3))
for name, f in acts.items():
    plt.plot(x, f(x), label=name)
plt.legend(); plt.grid(alpha=.3); plt.title("activation functions"); plt.show()

| 激活函数 | 公式 | 用在哪 |
|---|---|---|
| ReLU | $\max(0,x)$ | 经典 CNN、RLT 的 MLP actor |
| GELU | $x\Phi(x)$ | ViT、Gemma、BERT |
| SiLU | $x\cdot\text{sigmoid}(x)$ | π0 的时间 MLP、LLaMA 的 SwiGLU |
| Tanh | $\frac{e^x-e^{-x}}{e^x+e^{-x}}$ | 把输出压到 $[-1,1]$（SAC 的动作） |

---

## 2. 用 `nn.Module` 写一个 MLP

**规则**：在 `__init__` 里创建所有带参数的层（它们会被自动注册，`model.parameters()` 能找到）；在 `forward` 里写计算。

### 练习 1：实现一个可配置的 MLP

`MLP(in_dim, hidden_dim, out_dim, n_layers)`：共 `n_layers` 个隐藏层，每层 `Linear → 激活`，最后一个 `Linear` 输出。用 `nn.ModuleList` 或 `nn.Sequential` 存放各层。

> 为什么不能用普通 Python 列表存层？因为普通列表里的层**不会被注册**，`model.parameters()` 找不到它们，优化器就不会更新它们。这是一个经典 bug。

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, n_layers=2, act=nn.ReLU):
        super().__init__()
        # TODO: 构建 layers 列表：[Linear(in, h), act(), Linear(h, h), act(), ..., Linear(h, out)]
        #       然后 self.net = nn.Sequential(*layers)
        raise NotImplementedError

    def forward(self, x):
        return self.net(x)

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, n_layers=2, act=nn.ReLU):
        super().__init__()
        layers, d = [], in_dim
        for _ in range(n_layers):
            layers += [nn.Linear(d, hidden_dim), act()]
            d = hidden_dim
        layers.append(nn.Linear(d, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
# ✅ 自测
m = MLP(2, 64, 3, n_layers=2)
assert m(torch.randn(5, 2)).shape == (5, 3)
n_params = sum(p.numel() for p in m.parameters())
assert n_params == (2 * 64 + 64) + (64 * 64 + 64) + (64 * 3 + 3), n_params
print("通过 ✅ 参数量", n_params)

用 MLP 重新挑战上一课的螺旋数据：

In [ ]:
def make_spirals(n_per_class=200, K=3, noise=0.15, seed=0):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for k in range(K):
        r = np.linspace(0.05, 1, n_per_class)
        t = np.linspace(k * 4, (k + 1) * 4, n_per_class) + rng.normal(0, noise, n_per_class)
        X.append(np.stack([r * np.sin(t), r * np.cos(t)], 1)); y.append(np.full(n_per_class, k))
    return torch.tensor(np.concatenate(X), dtype=torch.float32), torch.tensor(np.concatenate(y))

torch.manual_seed(0)
X, y = make_spirals()
mlp = MLP(2, 128, 3, n_layers=2)
opt = torch.optim.Adam(mlp.parameters(), lr=1e-2)
for step in range(1500):
    loss = F.cross_entropy(mlp(X), y)
    opt.zero_grad(); loss.backward(); opt.step()
acc = (mlp(X).argmax(-1) == y).float().mean()
xx, yy = np.meshgrid(np.linspace(-1.2, 1.2, 200), np.linspace(-1.2, 1.2, 200))
with torch.no_grad():
    zz = mlp(torch.tensor(np.stack([xx.ravel(), yy.ravel()], 1), dtype=torch.float32)).argmax(-1).reshape(xx.shape)
plt.figure(figsize=(4, 4)); plt.contourf(xx, yy, zz, alpha=.3, cmap="brg"); plt.scatter(X[:, 0], X[:, 1], c=y, s=5, cmap="brg")
plt.title(f"MLP acc={acc:.2f}"); plt.show()

---

## 3. 过拟合与正则化

模型在训练集上表现好、在新数据上差，就是**过拟合**。机器人数据少（几十到几百条示教），过拟合是常态。

| 方法 | 做法 | PyTorch |
|---|---|---|
| 权重衰减 | 损失里加 $\lambda\|\mathbf{w}\|^2$，让权重别太大 | `AdamW(weight_decay=...)` |
| Dropout | 训练时随机把一部分神经元置 0 | `nn.Dropout(p)` |
| 数据增强 | 随机裁剪、颜色扰动 | openpi 训练时对图像做随机裁剪/旋转/颜色变化 |
| 早停 | 验证集损失不再下降就停 | 手动实现 |

> ⚠️ **Dropout 与 `train()/eval()`**：Dropout 和 BatchNorm 在训练和推理时行为不同。推理前**必须** `model.eval()`，训练前 `model.train()`。部署 VLA 时忘记 `eval()` 会让动作带随机噪声。

In [ ]:
drop = nn.Dropout(p=0.5)
x = torch.ones(8)
drop.train(); print("train 模式:", drop(x))   # 一半变 0，其余放大为 2（保持期望不变）
drop.eval();  print("eval  模式:", drop(x))   # 原样输出

---

## 4. 初始化

参数初始值太大 → 激活值爆炸；太小 → 信号消失。常用：

- **Xavier/Glorot**：$\text{Var}(w) = \frac{2}{n_{in}+n_{out}}$，适合 tanh；
- **Kaiming/He**：$\text{Var}(w) = \frac{2}{n_{in}}$，适合 ReLU（PyTorch `nn.Linear` 的默认值是它的一个变体）。
- **零初始化输出层**：一个特别有用的技巧。如果网络输出的是"对已有结果的修正量"（残差），把最后一层初始化为 0，**模型一开始就等价于"不修正"**。RLT 的残差 actor（`actor_output="residual"`）就是这样做的：一开始完全复现 VLA 的动作，再慢慢学修正。

In [ ]:
head = nn.Linear(64, 7)
nn.init.zeros_(head.weight); nn.init.zeros_(head.bias)
ref_action = torch.randn(1, 7)
print("零初始化的残差头：输出 = 参考动作 + 0 ->", torch.allclose(ref_action + head(torch.randn(1, 64)), ref_action))

---

## 5. 归一化层：BatchNorm、LayerNorm、RMSNorm

归一化让每层的输入保持在合适的尺度，使深层网络更容易训练。

- **BatchNorm**：对一个 batch 内**同一个特征**做归一化。依赖 batch 统计量，推理时用滑动平均。CNN 常用。
- **LayerNorm**：对**一个样本的所有特征**做归一化。不依赖 batch，Transformer 标配。

$$
\text{LN}(\mathbf{x}) = \boldsymbol\gamma\odot\frac{\mathbf{x}-\mu}{\sqrt{\sigma^2+\epsilon}} + \boldsymbol\beta,\quad \mu = \frac1D\sum_i x_i,\ \sigma^2 = \frac1D\sum_i(x_i-\mu)^2
$$

- **RMSNorm**：只除以均方根、不减均值，没有 $\beta$。更省计算，**Gemma（π0 的语言模型）、LLaMA 都用它**。π0.5 在它的基础上做了 **adaRMSNorm**：$\gamma$ 由时间 $t$ 生成（第 7 章）。

$$
\text{RMSNorm}(\mathbf{x}) = \boldsymbol\gamma\odot\frac{\mathbf{x}}{\sqrt{\frac1D\sum_i x_i^2+\epsilon}}
$$

### 练习 2：实现 LayerNorm 与 RMSNorm

In [ ]:
class MyLayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))   # γ
        self.bias = nn.Parameter(torch.zeros(dim))    # β

    def forward(self, x):
        # TODO: 在最后一维上求均值和（有偏）方差 var(unbiased=False)
        raise NotImplementedError


class MyRMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class MyLayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        mu = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.weight * (x - mu) / torch.sqrt(var + self.eps) + self.bias


class MyRMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.weight * x / rms

In [ ]:
# ✅ 自测
x = torch.randn(4, 10, 32) * 3 + 1
assert torch.allclose(MyLayerNorm(32)(x), nn.LayerNorm(32)(x), atol=1e-5)
r = MyRMSNorm(32)(x)
assert torch.allclose(r.pow(2).mean(-1), torch.ones(4, 10), atol=1e-3)   # 输出的均方值 ≈ 1
print("通过 ✅")

> 小知识：Gemma 的 RMSNorm 实现是 `x * (1 + weight)`，`weight` 初始化为 0，和这里的 `weight` 初始化为 1 等价。读源码时看到这种细节差异不要慌。

---

## 6. 优化器：SGD → Momentum → Adam → AdamW

- **SGD**：$\theta \leftarrow \theta - \eta\,g$
- **Momentum**：累积历史梯度方向，像小球滚下山，能冲过平坦区、抑制震荡。
- **Adam**：给每个参数一个**自适应学习率**。梯度一直很大的方向步长自动变小，梯度小的方向步长变大（还记得 1.04 课那个"椭圆碗"上的震荡吗？）。

Adam 的更新规则（$t$ 为步数，从 1 开始）：

$$
\begin{aligned}
m_t &= \beta_1 m_{t-1} + (1-\beta_1)g_t &\text{(一阶矩：梯度的滑动平均)}\\
v_t &= \beta_2 v_{t-1} + (1-\beta_2)g_t^2 &\text{(二阶矩：梯度平方的滑动平均)}\\
\hat m_t &= m_t/(1-\beta_1^t),\quad \hat v_t = v_t/(1-\beta_2^t) &\text{(偏差修正)}\\
\theta_t &= \theta_{t-1} - \eta\,\hat m_t/(\sqrt{\hat v_t}+\epsilon)
\end{aligned}
$$

**AdamW** 把权重衰减从梯度里"解耦"出来，直接 $\theta \leftarrow \theta - \eta\lambda\theta$。openpi 默认就用 AdamW。

### 练习 3：从零实现 Adam 的一步

In [ ]:
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.params = list(params)
        self.lr, (self.b1, self.b2), self.eps = lr, betas, eps
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0

    @torch.no_grad()
    def step(self):
        self.t += 1
        for p, m, v in zip(self.params, self.m, self.v):
            if p.grad is None:
                continue
            g = p.grad
            # TODO: 原地更新 m、v（用 m.mul_(...).add_(...)），做偏差修正，再更新 p
            raise NotImplementedError

    def zero_grad(self):
        for p in self.params:
            p.grad = None

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.params = list(params)
        self.lr, (self.b1, self.b2), self.eps = lr, betas, eps
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0

    @torch.no_grad()
    def step(self):
        self.t += 1
        for p, m, v in zip(self.params, self.m, self.v):
            if p.grad is None:
                continue
            g = p.grad
            m.mul_(self.b1).add_((1 - self.b1) * g)
            v.mul_(self.b2).add_((1 - self.b2) * g * g)
            m_hat = m / (1 - self.b1 ** self.t)
            v_hat = v / (1 - self.b2 ** self.t)
            p.sub_(self.lr * m_hat / (v_hat.sqrt() + self.eps))

    def zero_grad(self):
        for p in self.params:
            p.grad = None

In [ ]:
# ✅ 自测：和 torch.optim.Adam 跑 20 步，参数应该完全一致
torch.manual_seed(0)
net1 = MLP(2, 16, 3); net2 = MLP(2, 16, 3); net2.load_state_dict(net1.state_dict())
o1 = MyAdam(net1.parameters(), lr=1e-2); o2 = torch.optim.Adam(net2.parameters(), lr=1e-2)
for _ in range(20):
    for net, o in [(net1, o1), (net2, o2)]:
        loss = F.cross_entropy(net(X), y); o.zero_grad(); loss.backward(); o.step()
for p1, p2 in zip(net1.parameters(), net2.parameters()):
    assert torch.allclose(p1, p2, atol=1e-5)
print("通过 ✅ 你的 Adam 与 PyTorch 完全一致")

---

## 7. 学习率调度与梯度裁剪

大模型训练的标准配方：**先线性预热（warmup），再余弦衰减（cosine decay）**。预热防止一开始梯度很乱时步子太大；衰减让后期精细收敛。openpi 的默认配置是 `CosineDecaySchedule(warmup_steps=1000, peak_lr=2.5e-5, decay_steps=30000, decay_lr=2.5e-6)`。

**梯度裁剪**：如果梯度的总范数超过阈值，就按比例缩小。防止偶尔一个"坏 batch"把模型带飞。openpi 默认 `clip_gradient_norm=1.0`，PyTorch 中是 `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)`。

### 练习 4：预热 + 余弦衰减

$$
\eta(s) = \begin{cases}
\eta_{peak}\cdot\frac{s+1}{W} & s < W\\
\eta_{end} + \frac12(\eta_{peak}-\eta_{end})\left(1+\cos\left(\pi\cdot\frac{s-W}{S-W}\right)\right) & W\le s < S\\
\eta_{end} & s\ge S
\end{cases}
$$

In [ ]:
import math

def warmup_cosine(step, warmup, total, peak_lr, end_lr):
    # TODO: 按上面的分段公式实现
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
import math

def warmup_cosine(step, warmup, total, peak_lr, end_lr):
    if step < warmup:
        return peak_lr * (step + 1) / warmup
    if step >= total:
        return end_lr
    progress = (step - warmup) / (total - warmup)
    return end_lr + 0.5 * (peak_lr - end_lr) * (1 + math.cos(math.pi * progress))

In [ ]:
# ✅ 自测（openpi 的默认超参数）
lrs = [warmup_cosine(s, 1000, 30000, 2.5e-5, 2.5e-6) for s in range(32000)]
assert abs(lrs[999] - 2.5e-5) < 1e-12 and abs(lrs[-1] - 2.5e-6) < 1e-12
assert abs(lrs[15500] - (2.5e-6 + 0.5 * (2.5e-5 - 2.5e-6))) < 1e-9
plt.figure(figsize=(5, 2.5)); plt.plot(lrs); plt.title("warmup + cosine decay (openpi default)"); plt.xlabel("step"); plt.show()
print("通过 ✅")

---

## 小结

| 技巧 | 作用 | openpi / RLT 中的对应 |
|---|---|---|
| 非线性激活 | 让网络能拟合复杂函数 | GELU（Gemma）、SiLU（时间 MLP） |
| 零初始化残差头 | 一开始"不改变"已有输出 | RLT 残差 actor |
| LayerNorm / RMSNorm | 稳定深层网络 | Gemma 用 RMSNorm；π0.5 用 adaRMSNorm |
| Dropout / 权重衰减 / 数据增强 | 防过拟合 | AdamW + 图像增强 |
| Adam / AdamW | 自适应学习率 | 默认优化器 |
| warmup + cosine | 稳定开局、精细收尾 | `CosineDecaySchedule` |
| 梯度裁剪 | 防止梯度爆炸 | `clip_gradient_norm=1.0` |